# Implement a program for retrieval of documents using inverted files.
**Computer Laboratory II (417526) – Information Retrieval – Practical 2**

Install once (Linux terminal):
```
pip install nltk pandas
```

In [ ]:
import os, re, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
warnings.filterwarnings("ignore")

DATA_DIR = "data"
def get_file(name, url):
    # Return path to data/<name>; download it first if it is not there yet.
    os.makedirs(DATA_DIR, exist_ok=True)
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        print("Downloading", url)
        urllib.request.urlretrieve(url, path)
    return path

In [ ]:
import nltk
# NLTK resources (needs internet once). The notebook still works without them using the fallbacks below.
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print("could not download", pkg, "->", e)

In [ ]:
from nltk.corpus import stopwords

# Offline copy of NLTK's English stop-word list (used only if the NLTK data cannot be downloaded)
NLTK_STOP_FALLBACK = set('''i me my myself we our ours ourselves you you're you've you'll you'd your yours yourself
yourselves he him his himself she she's her hers herself it it's its itself they them their theirs themselves what which
who whom this that that'll these those am is are was were be been being have has had having do does did doing a an the
and but if or because as until while of at by for with about against between into through during before after above below
to from up down in out on off over under again further then once here there when where why how all any both each few more
most other some such no nor not only own same so than too very s t can will just don don't should should've now d ll m o re
ve y ain aren aren't couldn couldn't didn didn't doesn doesn't hadn hadn't hasn hasn't haven haven't isn isn't ma mightn
mightn't mustn mustn't needn needn't shan shan't shouldn shouldn't wasn wasn't weren weren't won won't wouldn wouldn't'''.split())

text = '''Natural Language Processing (NLP) is a field of Artificial Intelligence. It helps computers read, process
and understand human language! Dr. Smith studies how machines are learning to translate languages better.
Search engines apply stemming for retrieving documents that match the queries of users.'''

# --- the stop-word list (NLTK, with an offline fallback) ---
try:
    STOP = set(stopwords.words("english"))
except LookupError:
    STOP = set(NLTK_STOP_FALLBACK)
    print("NLTK stopwords not available -> using the built-in copy of NLTK's English stop-word list")

## Practical 2 – Retrieval of documents using inverted files
An **inverted index** maps every term to the places where it occurs (like a hash map from *word → documents/lines*). To answer a query we look up the terms in the index instead of scanning every document.

**Steps:** read documents → remove punctuation & lower-case → tokenize → remove stop words → build `term → {doc: [positions]}` → search (single term, AND, OR) → rank by term frequency.

In [ ]:
from collections import defaultdict

# --- Part A: the manual's example, line level index of file.txt ---
with open("file.txt", "w") as f:
    f.write("This is the first word.\nThis is the second text, Hello! How are you?\nThis is the third, this is it now.\n")

with open("file.txt", encoding="utf8") as f:
    lines = f.read().splitlines()
print("Number of lines in file is:", len(lines))
print(lines)

def tokenize(s):
    # remove punctuation, lower-case, split, drop stop words
    s = re.sub(r"[^\w\s]", " ", s.lower())
    return [w for w in s.split() if w not in STOP]

def build_inverted_index(docs):
    # docs: {doc_id: text}. Returns {term: {doc_id: [positions]}}
    index = defaultdict(lambda: defaultdict(list))
    for doc_id, txt in docs.items():
        for pos, term in enumerate(tokenize(txt)):
            index[term][doc_id].append(pos)
    return index

line_index = build_inverted_index({i + 1: l for i, l in enumerate(lines)})
print("\nWord -> line numbers:")
print({term: sorted(postings) for term, postings in line_index.items()})

In [ ]:
# --- Part B: a small document collection ---
docs = {
    "doc1": "Information retrieval is the process of finding relevant documents for a user query.",
    "doc2": "Web mining applies data mining techniques to discover patterns from the web.",
    "doc3": "A search engine uses an inverted index to retrieve documents quickly.",
    "doc4": "Stemming and stop word removal are used to pre-process documents before indexing.",
    "doc5": "PageRank ranks web pages using the link structure of the web.",
}
index = build_inverted_index(docs)
print("Vocabulary size:", len(index))
for term in ["documents", "web", "inverted", "index"]:
    print(f"{term:10s} ->", {d: len(p) for d, p in index[term].items()})

def search(query, mode="AND"):
    # Boolean retrieval (AND / OR) on the inverted index, ranked by total term frequency.
    terms = tokenize(query)
    if not terms:
        return []
    postings = [set(index[t].keys()) if t in index else set() for t in terms]
    hits = set.intersection(*postings) if mode == "AND" else set.union(*postings)
    score = {d: sum(len(index[t][d]) for t in terms if t in index and d in index[t]) for d in hits}
    return sorted(score.items(), key=lambda x: (-x[1], x[0]))

for q, m in [("web mining", "AND"), ("documents index", "AND"), ("documents web", "OR"), ("quantum", "AND")]:
    print(f"\nQuery '{q}' [{m}] ->", search(q, m))
print()
for d, s in search("web", "OR"):
    print(d, "|", docs[d])